# facts


In [ ]:
"""
Popula los 6 hechos del modelo dimensional (esquema dw) leyendo de staging
(stage.*) y uniendo con las dimensiones (dw.dim_*) para obtener los surrogate
keys. El contrato de columnas/PK es scripts/dw/ddl_oracle.sql — la salida coincide
en nombres, orden y GRANO (PK) con ese DDL.

Reglas Kimball aplicadas:
  - FKs en hechos: NUNCA nulas. coalesce(id, GEN_ID) con la fila genérica de cada
    dimensión (id=-1, o id=0 para género Desconocido).
  - Las medidas se AGREGAN al grano de la PK del DDL (clave: evita violar la PK):
      · fact_ine     → defuncion = COUNT de defunciones por combinación dimensional
      · fact_panama  → defunciones = SUM por combinación dimensional
      · fact_who_deaths → se toma la fila "Todas las edades" (id_etario=99): trae el
        total y las tasas válidas (sumar tasas sería incorrecto).
      · fact_who_population → solo el detalle (sexo H/M, edades 1-19 y 98); se
        descartan los totales pre-agregados (sexo 'all', edad 99) para que SUM
        sea aditivo sin doble conteo.
  - Causa: join por clave natural = CIE-10 individual (si válido) ELSE GBD. WHO
    deaths engancha por su super-categoría icd10_group (causa sintética WHO_*).

Hechos: fact_ihme, fact_ine, fact_mspas, fact_panama, fact_who_deaths,
        fact_who_population.
"""

from pyspark.sql import functions as F, DataFrame
from pyspark.sql.types import IntegerType, StringType

SCHEMA_DW    = "semi2.dm_mortality"
SCHEMA_STAGE = "stage"

GEN_ID_NEG   = -1   # genérico dim_causa, dim_geografia, dim_ine_perfil, dim_tiempo
GEN_ID_ZERO  = 0    # dim_genero (Desconocido)
GEN_ETARIO_UNK = 98 # dim_etario "No especificada"

CIE10_RE = r"^[A-Z][0-9]{2}[0-9A-Z]?$"

# icd10_group de WHO → código de causa sintético (debe coincidir con dimensions.py)
WHO_ICD10_CAT_CODE = {
    "communicable_maternal_perinatal_and_nutritional_conditions": "WHO_I",
    "noncommunicable_diseases": "WHO_II",
    "injuries": "WHO_III",
    "ill_defined_diseases": "WHO_IV",
}


def _broadcast(df: DataFrame) -> DataFrame:
    return F.broadcast(df)


def _save(df: DataFrame, table: str) -> None:
    full = f"{SCHEMA_DW}.{table}"
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(full)
    print(f"[DW] {full} — {df.count()} filas")


# ── Canonicalización INE (idéntica a dimensions.canon_ine_perfil) ────────────

def _canon_ine_perfil(df: DataFrame) -> DataFrame:
    ec = F.trim(F.regexp_replace(F.col("estado_civil"), r"\(a\)\s*$", ""))
    esc = F.when(F.col("escolaridad") == "Básico",  F.lit("Básica")) \
           .when(F.col("escolaridad") == "Ninguno", F.lit("Ninguna")) \
           .otherwise(F.col("escolaridad"))
    return df.withColumn("estado_civil", ec).withColumn("escolaridad", esc)


# ── Normalización de sexo → valores canónicos de dim_genero ──────────────────

def _normalize_sexo_ihme(df: DataFrame) -> DataFrame:
    return df.withColumn("_sexo_norm",
        F.when(F.col("sexo") == "Ambos",  F.lit("Ambos"))
         .when(F.col("sexo") == "Hombre", F.lit("Hombre"))
         .when(F.col("sexo") == "Mujer",  F.lit("Mujer"))
         .otherwise(F.lit("Desconocido")))


def _normalize_sexo_ine(df: DataFrame) -> DataFrame:
    return df.withColumn("_sexo_norm",
        F.when(F.col("sexo") == "Hombre", F.lit("Hombre"))
         .when(F.col("sexo") == "Mujer",  F.lit("Mujer"))
         .otherwise(F.lit("Desconocido")))


def _normalize_sexo_panama(df: DataFrame) -> DataFrame:
    return df.withColumn("_sexo_norm",
        F.when(F.col("sexo") == "Hombres", F.lit("Hombre"))
         .when(F.col("sexo") == "Mujeres", F.lit("Mujer"))
         .otherwise(F.lit("Desconocido")))


def _normalize_sexo_who(df: DataFrame) -> DataFrame:
    return df.withColumn("_sexo_norm",
        F.when(F.col("sex") == "male",   F.lit("Hombre"))
         .when(F.col("sex") == "female", F.lit("Mujer"))
         .when(F.col("sex") == "all",    F.lit("Ambos"))
         .otherwise(F.lit("Desconocido")))


# ── Joins con dimensiones ────────────────────────────────────────────────────

def _join_geografia(df: DataFrame, dim_geo: DataFrame,
                    pais_col: str = "pais_iso3",
                    dep_col: str = None, mun_col: str = None) -> DataFrame:
    cond = df[pais_col] == dim_geo.pais_iso3
    if dep_col:
        cond = cond & df[dep_col].eqNullSafe(dim_geo.dep_ocurrencia)
    else:
        cond = cond & (dim_geo.dep_ocurrencia == "No aplica")
    if mun_col:
        cond = cond & df[mun_col].eqNullSafe(dim_geo.mun_ocurrencia)
    else:
        cond = cond & (dim_geo.mun_ocurrencia == "No aplica")

    df = df.join(_broadcast(dim_geo), cond, "left")
    df = df.withColumn("id_geografia", F.coalesce(F.col("id_geografia"), F.lit(GEN_ID_NEG)))
    return df.drop("pais_iso3", "pais_nombre", "dep_ocurrencia",
                   "mun_ocurrencia")


def _join_tiempo(df: DataFrame, dim_tpo: DataFrame,
                 anio_col: str = "anio", mes_col: str = None) -> DataFrame:
    cond = df[anio_col] == dim_tpo.anio_ocurrencia
    if mes_col:
        cond = cond & df[mes_col].eqNullSafe(dim_tpo.mes_ocurrencia)
    else:
        cond = cond & (dim_tpo.mes_ocurrencia_num == GEN_ID_ZERO)

    df = df.join(_broadcast(dim_tpo), cond, "left")
    df = df.withColumn("id_tiempo", F.coalesce(F.col("id_tiempo"), F.lit(GEN_ID_NEG)))
    return df.drop("anio_ocurrencia", "mes_ocurrencia", "mes_ocurrencia_num",
                   "es_pre_covid")


def _join_genero(df: DataFrame, dim_gen: DataFrame) -> DataFrame:
    df = df.join(_broadcast(dim_gen), df["_sexo_norm"] == dim_gen.sexo_codigo, "left")
    df = df.withColumn("id_genero", F.coalesce(F.col("id_genero"), F.lit(GEN_ID_ZERO)))
    return df.drop("sexo_codigo", "sexo_nombre", "_sexo_norm")


def _causa_lookup(dim_cau: DataFrame) -> DataFrame:
    """Reconstruye la clave natural de causa desde dim_causa:
       causa_nk = cie-10_code (si != 'N/A') ELSE gbd_code (si != 'N/A')."""
    nk = F.when(F.col("`cie-10_code`") != "N/A", F.col("`cie-10_code`")) \
          .when(F.col("gbd_code") != "N/A", F.col("gbd_code")) \
          .otherwise(F.lit(None).cast(StringType()))
    return (dim_cau.select("id_causa", nk.alias("causa_nk"))
                   .filter(F.col("causa_nk").isNotNull()))


def _join_causa(df: DataFrame, causa_lkp: DataFrame) -> DataFrame:
    """Une por 'causa_nk' (ya presente en df). id_causa nunca NULL → -1."""
    df = df.join(_broadcast(causa_lkp), "causa_nk", "left")
    df = df.withColumn("id_causa", F.coalesce(F.col("id_causa"), F.lit(GEN_ID_NEG)))
    return df.drop("causa_nk")


def _causa_nk_cie_gbd(cie_col: str, gbd_col: str):
    """Expr: CIE-10 individual válido ELSE gbd (limpiando 'null' literal)."""
    cie = F.when(
        F.col(cie_col).isNotNull() & F.upper(F.trim(F.col(cie_col))).rlike(CIE10_RE),
        F.upper(F.trim(F.col(cie_col)))
    ).otherwise(F.lit(None).cast(StringType()))
    gbd = F.when(F.col(gbd_col) == "null", F.lit(None).cast(StringType())) \
           .otherwise(F.col(gbd_col))
    return F.coalesce(cie, gbd)


def _join_source(df: DataFrame, dim_src: DataFrame, source_name: str) -> DataFrame:
    df = df.join(
        _broadcast(dim_src.filter(F.col("source") == source_name).select("id_source", "source")),
        F.lit(True), "left",
    )
    return df.drop("source")


def _coalesce_etario(df: DataFrame) -> DataFrame:
    return df.withColumn("id_etario", F.coalesce(F.col("id_etario"), F.lit(GEN_ETARIO_UNK)))


# ═══════════════════════════════════════════════════════════════════════
# FACT_IHME  (grano ya único: país×sexo×causa×año×perfil; sin agregación)
# ═══════════════════════════════════════════════════════════════════════

def build_fact_ihme(dims: dict) -> DataFrame:
    print("--- FACT_IHME ---")
    stage_df = _normalize_sexo_ihme(spark.read.table(f"{SCHEMA_STAGE}.ihme"))
    # IHME solo trae GBD (sin CIE-10): causa_nk = gbd_code (limpiando 'null').
    stage_df = stage_df.withColumn(
        "causa_nk",
        F.when(F.col("gbd_code") == "null", F.lit(None).cast(StringType()))
         .otherwise(F.col("gbd_code")),
    )

    df = _join_geografia(stage_df, dims["dim_geografia"], "pais_iso3")
    df = _join_genero(df, dims["dim_genero"])
    df = _join_causa(df, _causa_lookup(dims["dim_causa"]))
    df = _join_tiempo(df, dims["dim_tiempo"], "anio")
    df = _join_source(df, dims["dim_source"], "IHME GBD 2023")

    prof = dims["dim_ihme_perfil"]
    df = df.join(_broadcast(prof),
                 [df["medida"] == prof.medida, df["metrica"] == prof.metrica],
                 "left").drop("medida", "metrica")

    return df.select(
        F.col("valor"),
        F.col("limite_inferior"),
        F.col("limite_superior"),
        F.col("id_geografia").cast(IntegerType()),
        F.col("id_genero").cast(IntegerType()),
        F.col("id_causa").cast(IntegerType()),
        F.col("id_tiempo").cast(IntegerType()),
        F.col("id_source").cast(IntegerType()),
        F.col("id_ihme_perfil").cast(IntegerType()),
    )


# ═══════════════════════════════════════════════════════════════════════
# FACT_INE  (agrega: defuncion = COUNT por grano de la PK)
# ═══════════════════════════════════════════════════════════════════════

def build_fact_ine(dims: dict) -> DataFrame:
    print("--- FACT_INE ---")
    stage_df = spark.read.table(f"{SCHEMA_STAGE}.ine")
    stage_df = _canon_ine_perfil(stage_df)
    stage_df = _normalize_sexo_ine(stage_df)
    # Coalesce NULLs → 'Desconocido' como en dim_ine_perfil para que el join calce
    stage_df = (stage_df
        .withColumn("estado_civil",      F.coalesce(F.col("estado_civil"),      F.lit("Desconocido")))
        .withColumn("escolaridad",       F.coalesce(F.col("escolaridad"),       F.lit("Desconocido")))
        .withColumn("asistencia_medica", F.coalesce(F.col("asistencia_medica"), F.lit("Desconocido")))
        .withColumn("tipo_ocurrencia",   F.coalesce(F.col("tipo_ocurrencia"),   F.lit("Desconocido"))))
    stage_df = stage_df.withColumn("causa_nk", _causa_nk_cie_gbd("cie10_code", "gbd_code"))

    df = _join_geografia(stage_df, dims["dim_geografia"],
                         "pais_iso3", "dep_ocurrencia", "mun_ocurrencia")
    df = _join_genero(df, dims["dim_genero"])
    df = _join_causa(df, _causa_lookup(dims["dim_causa"]))
    df = _join_tiempo(df, dims["dim_tiempo"], "anio_ocurrencia", "mes_ocurrencia")
    df = _join_source(df, dims["dim_source"], "INE Guatemala")
    df = _coalesce_etario(df)

    # dim_ine_perfil: FK por las 4 categorías canonicalizadas → -1 si no calza.
    prof = dims["dim_ine_perfil"]
    df = df.join(
        _broadcast(prof),
        [df["estado_civil"].eqNullSafe(prof.estado_civil),
         df["escolaridad"].eqNullSafe(prof.escolaridad),
         df["asistencia_medica"].eqNullSafe(prof.asistencia_medica),
         df["tipo_ocurrencia"].eqNullSafe(prof.tipo_ocurrencia)],
        "left",
    ).drop("estado_civil", "escolaridad", "asistencia_medica", "tipo_ocurrencia")
    df = df.withColumn("id_ine_perfil", F.coalesce(F.col("id_ine_perfil"), F.lit(GEN_ID_NEG)))

    # Agregar al grano de la PK: defuncion = conteo de defunciones individuales.
    pk = ["id_geografia", "id_tiempo", "id_source", "id_causa",
          "id_etario", "id_genero", "id_ine_perfil"]
    df = df.groupBy(*[F.col(c).cast(IntegerType()).alias(c) for c in pk]) \
           .agg(F.count(F.lit(1)).cast(IntegerType()).alias("defuncion"))

    return df.select(
        F.col("defuncion"),
        "id_geografia", "id_tiempo", "id_source", "id_causa",
        "id_etario", "id_genero", "id_ine_perfil",
    )


# ═══════════════════════════════════════════════════════════════════════
# FACT_MSPAS  (general + tasa; grano geo×source×año, ya único)
# ═══════════════════════════════════════════════════════════════════════

def build_fact_mspas(dims: dict) -> DataFrame:
    print("--- FACT_MSPAS ---")
    dfs = []

    gen = f"{SCHEMA_STAGE}.mspas_mortalidad_general"
    if spark.catalog.tableExists(gen):
        d = spark.read.table(gen)
        d = _join_geografia(d, dims["dim_geografia"], "pais_iso3")
        d = _join_tiempo(d, dims["dim_tiempo"], "anio")
        d = _join_source(d, dims["dim_source"], "MSPAS Guatemala")
        dfs.append(d.select(
            F.col("defunciones").cast("bigint"),
            F.lit(None).cast("double").alias("tasa_por_100k"),
            F.col("id_geografia").cast(IntegerType()),
            F.col("id_source").cast(IntegerType()),
            F.col("id_tiempo").cast(IntegerType()),
        ))

    tasa = f"{SCHEMA_STAGE}.mspas_tasa"
    if spark.catalog.tableExists(tasa):
        d = spark.read.table(tasa)
        d = _join_geografia(d, dims["dim_geografia"], "pais_iso3")
        d = _join_tiempo(d, dims["dim_tiempo"], "anio")
        d = _join_source(d, dims["dim_source"], "MSPAS Guatemala")
        dfs.append(d.select(
            F.lit(None).cast("bigint").alias("defunciones"),
            F.col("tasa_por_100k").cast("double"),
            F.col("id_geografia").cast(IntegerType()),
            F.col("id_source").cast(IntegerType()),
            F.col("id_tiempo").cast(IntegerType()),
        ))

    if not dfs:
        print("[WARN] fact_mspas: ninguna tabla stage encontrada.")
        return None

    if len(dfs) == 2:
        return (
            dfs[0].alias("a")
            .join(dfs[1].alias("b"),
                  [F.col("a.id_geografia") == F.col("b.id_geografia"),
                   F.col("a.id_source")    == F.col("b.id_source"),
                   F.col("a.id_tiempo")    == F.col("b.id_tiempo")],
                  "full_outer")
            .select(
                F.coalesce(F.col("a.defunciones"),   F.col("b.defunciones")).alias("defunciones"),
                F.coalesce(F.col("a.tasa_por_100k"), F.col("b.tasa_por_100k")).alias("tasa_por_100k"),
                F.coalesce(F.col("a.id_geografia"),  F.col("b.id_geografia")).alias("id_geografia"),
                F.coalesce(F.col("a.id_source"),     F.col("b.id_source")).alias("id_source"),
                F.coalesce(F.col("a.id_tiempo"),     F.col("b.id_tiempo")).alias("id_tiempo"),
            )
        )
    return dfs[0]


# ═══════════════════════════════════════════════════════════════════════
# FACT_PANAMA  (agrega: defunciones = SUM por grano de la PK)
# 'grupo' (095 externas, U00-U85 especiales) y 'simple' (003-094) son conjuntos
# de causas DISJUNTOS → se conservan ambos (no hay doble conteo); el SUM solo
# colapsa las sub-filas del mismo código (p.ej. las 16 sub-causas de 095).
# ═══════════════════════════════════════════════════════════════════════

def build_fact_panama(dims: dict) -> DataFrame:
    print("--- FACT_PANAMA ---")
    table = f"{SCHEMA_STAGE}.panama"
    if not spark.catalog.tableExists(table):
        print("[WARN] fact_panama: stage.panama no existe.")
        return None

    stage_df = _normalize_sexo_panama(spark.read.table(table))
    stage_df = stage_df.withColumn("causa_nk",
        F.coalesce(_causa_nk_cie_gbd("cie10_code", "gbd_code"), F.col("causa_codigo")))

    df = _join_geografia(stage_df, dims["dim_geografia"], "pais_iso3")
    df = _join_genero(df, dims["dim_genero"])
    df = _join_causa(df, _causa_lookup(dims["dim_causa"]))
    df = _join_tiempo(df, dims["dim_tiempo"], "anio")
    df = _join_source(df, dims["dim_source"], "INEC Panamá")
    df = _coalesce_etario(df)

    pk = ["id_etario", "id_causa", "id_genero", "id_source", "id_tiempo", "id_geografia"]
    df = df.groupBy(*[F.col(c).cast(IntegerType()).alias(c) for c in pk]) \
           .agg(F.sum(F.col("defunciones")).cast("bigint").alias("defunciones"))

    total = df.count()
    nulo = df.filter(F.col("id_causa") == GEN_ID_NEG).count()
    if total:
        print(f"  Panamá: {nulo}/{total} filas con id_causa=-1 ({100*nulo/total:.1f}%)")

    return df.select(
        "id_etario", "id_causa", "id_genero", "id_source", "id_tiempo", "id_geografia",
        F.col("defunciones"),
    )


# ═══════════════════════════════════════════════════════════════════════
# FACT_WHO_DEATHS  (grano sin edad → fila "Todas las edades" id_etario=99,
# que ya trae el total y las tasas correctas)
# ═══════════════════════════════════════════════════════════════════════

def build_fact_who_deaths(dims: dict) -> DataFrame:
    print("--- FACT_WHO_DEATHS ---")
    table = f"{SCHEMA_STAGE}.who_deaths_by_age_group_gtm"
    if not spark.catalog.tableExists(table):
        print("[WARN] fact_who_deaths: tabla no encontrada.")
        return None

    stage_df = spark.read.table(table).filter(F.col("id_etario") == 99)
    stage_df = _normalize_sexo_who(stage_df)
    cat_map = F.create_map(
        *[x for k, v in WHO_ICD10_CAT_CODE.items() for x in (F.lit(k), F.lit(v))]
    )
    stage_df = stage_df.withColumn("causa_nk", cat_map[F.col("icd10_group")])

    df = _join_geografia(stage_df, dims["dim_geografia"], "country_code")
    df = _join_genero(df, dims["dim_genero"])
    df = _join_causa(df, _causa_lookup(dims["dim_causa"]))
    df = _join_tiempo(df, dims["dim_tiempo"], "year")
    df = _join_source(df, dims["dim_source"], "WHO Mortality")

    return df.select(
        F.col("number").alias("number"),
        F.col("percentage_of_cause_specific_deaths_out_of_total_deaths")
          .alias("prc_cause_specific_deaths_out_of_total_deaths"),
        F.col("age_standardized_death_rate_per_100_000_standard_population")
          .alias("age_std_death_rate_per_100k_std_population"),
        F.col("death_rate_per_100_000_population")
          .alias("death_rate_per_100k_population"),
        F.col("id_source").cast(IntegerType()),
        F.col("id_geografia").cast(IntegerType()),
        F.col("id_causa").cast(IntegerType()),
        F.col("id_tiempo").cast(IntegerType()),
        F.col("id_genero").cast(IntegerType()),
    )


# ═══════════════════════════════════════════════════════════════════════
# FACT_WHO_POPULATION  (solo detalle: sexo H/M, edades 1-19 y 98; se descartan
# los totales pre-agregados sexo='all' y edad=99 → población aditiva sin duplicar)
# ═══════════════════════════════════════════════════════════════════════

def build_fact_who_population(dims: dict) -> DataFrame:
    print("--- FACT_WHO_POPULATION ---")
    table = f"{SCHEMA_STAGE}.who_population_distribution_gtm"
    if not spark.catalog.tableExists(table):
        print("[WARN] fact_who_population: tabla no encontrada.")
        return None

    stage_df = (
        spark.read.table(table)
        .filter(F.col("sex").isin("male", "female"))
        .filter(F.col("id_etario") != 99)
    )
    stage_df = _normalize_sexo_who(stage_df)

    df = _join_geografia(stage_df, dims["dim_geografia"], "country_code")
    df = _join_genero(df, dims["dim_genero"])
    df = _join_tiempo(df, dims["dim_tiempo"], "year")
    df = _join_source(df, dims["dim_source"], "WHO Mortality")
    df = _coalesce_etario(df)

    return df.select(
        F.col("population"),
        F.col("id_source").cast(IntegerType()),
        F.col("id_tiempo").cast(IntegerType()),
        F.col("id_geografia").cast(IntegerType()),
        F.col("id_genero").cast(IntegerType()),
        F.col("id_etario").cast(IntegerType()),
    )


# ═══════════════════════════════════════════════════════════════════════
# EJECUCIÓN
# ═══════════════════════════════════════════════════════════════════════

def main():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_DW}")

    dims = {}
    for t in ["dim_etario", "dim_source", "dim_genero", "dim_causa",
              "dim_geografia", "dim_tiempo", "dim_ihme_perfil", "dim_ine_perfil"]:
        full = f"{SCHEMA_DW}.{t}"
        if spark.catalog.tableExists(full):
            dims[t] = spark.read.table(full)
            print(f"  Cargada {full} ({dims[t].count()} filas)")
        else:
            print(f"  [WARN] {full} no existe — ejecutá dimensions.py primero")
            return

    print("=== Poblando hechos ===")

    for name, fn in [
        ("fact_ihme",           build_fact_ihme),
        ("fact_ine",            build_fact_ine),
        ("fact_mspas",          build_fact_mspas),
        ("fact_panama",         build_fact_panama),
        ("fact_who_deaths",     build_fact_who_deaths),
        ("fact_who_population", build_fact_who_population),
    ]:
        df = fn(dims)
        if df is not None:
            _save(df, name)

    print("=== Hechos poblados ===")
    for t in ["fact_ihme", "fact_ine", "fact_mspas",
              "fact_panama", "fact_who_deaths", "fact_who_population"]:
        full = f"{SCHEMA_DW}.{t}"
        if spark.catalog.tableExists(full):
            display(spark.read.table(full).limit(10))


if __name__ == "__main__":
    main()
else:
    main()
